# 04 — Layer 2B Risk / Regime Engine

This notebook builds the risk and regime machinery that supports ETF strategy decisions.

**What this notebook does**

- computes rolling volatility, downside risk, drawdown, beta, and covariance inputs
- turns Layer 1 regime features plus additional cross-asset diagnostics into a transparent regime engine
- produces reusable defensive overlays and risk-budget multipliers
- tests whether momentum, reversal, and simple risk-on sleeves behave differently across regimes
- saves Layer 2B outputs in a form that later Layer 3 portfolio construction can reuse directly


## 1. Timing Convention and Design Scope

Layer 2B is still part of the same weekly research pipeline.

**Timing rules**

- price-based risk estimates are lagged by one weekly bar when used for decisions
- external macro / sentiment features are used only in their already-lagged Layer 1 form, or lagged again when newly built here
- any defensive overlay multiplier saved by this notebook is intended to be *tradable*, not contemporaneous

**What belongs here**

- risk estimation
- regime classification
- broad stress / calm / trend-friendly labels
- defensive overlay logic
- covariance inputs that later allocation notebooks will need

**What does not belong here**

- a full optimizer
- bespoke overfit regime models
- hidden complexity that makes a backtest look smarter than it really is


In [ ]:
from __future__ import annotations

import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from sklearn.covariance import LedoitWolf
except Exception:
    LedoitWolf = None

warnings.filterwarnings("ignore")
pd.options.display.float_format = "{:,.4f}".format
plt.rcParams["figure.figsize"] = (13, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
np.random.seed(7)


In [ ]:
PROJECT_DIR = Path.cwd()
DATA_ROOT = PROJECT_DIR / "data"

DATA_HUB_DIR = DATA_ROOT / "01_data_hub"
LAYER1_DIR = DATA_ROOT / "02_layer1_signals"
OUTPUT_DIR = DATA_ROOT / "04_layer2b_risk_regime_engine"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_HUB_CANDIDATES = [
    DATA_HUB_DIR,
    PROJECT_DIR / "data" / "processed",
    PROJECT_DIR.parent / "data" / "processed",
    PROJECT_DIR.parent / "data2" / "processed",
]
LAYER1_CANDIDATES = [
    LAYER1_DIR,
    PROJECT_DIR / "data" / "processed",
    PROJECT_DIR.parent / "data" / "processed",
    PROJECT_DIR.parent / "data2" / "processed",
]

REQUIRED_DATA_HUB = [
    "weekly_prices.csv",
    "weekly_returns.csv",
    "macro_weekly.csv",
    "vix_term_structure.csv",
    "universe.json",
]
OPTIONAL_DATA_HUB = [
    "daily_returns.csv",
    "proxy_mapping.json",
    "benchmark_returns_weekly.csv",
    "market_proxy_weekly.csv",
    "universe_metadata.csv",
]
OPTIONAL_LAYER1 = [
    "signal_xsmom.csv",
    "signal_reversal.csv",
    "signal_multi_horizon_mom.csv",
    "signal_quality.csv",
    "regime_features.csv",
]

WEEKS_PER_YEAR = 52
TRADING_DAYS_PER_YEAR = 252
SIGNAL_LAG_WEEKS = 1
EXTERNAL_DATA_LAG_WEEKS = 1
VOL_WINDOWS = (13, 26, 52)
DAILY_VOL_WINDOWS = (21, 63)
DRAWDOWN_WINDOW = 52
BETA_WINDOW = 52
BETA_MIN_PERIODS = 26
COVARIANCE_WINDOW = 26
REGIME_Z_WINDOW = 52
DEFAULT_COST_BPS = 10
REGIME_TARGET_VOL_ANN = 0.12
REGIME_TARGET_VOL_FLOOR = 0.25
REGIME_TARGET_VOL_CEIL = 1.25

print("Project directory:", PROJECT_DIR.resolve())
print("Data hub directory:", DATA_HUB_DIR.resolve())
print("Layer 1 directory:", LAYER1_DIR.resolve())
print("Layer 2B output directory:", OUTPUT_DIR.resolve())


In [ ]:
# Small technical-debt note: a few basic IO / metric helpers still mirror Layer 2A / Layer 3 so each notebook stays self-contained. A tiny shared utility module would be a clean future cleanup, but this pass keeps behavior stable.
def flatten_columns(df):
    df = df.copy()
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = ["_".join([str(x) for x in col if x not in [None, ""]]) for col in df.columns]
    df.columns = [str(c) for c in df.columns]
    df.columns.name = None
    df.index = pd.to_datetime(df.index).tz_localize(None)
    df.index.name = "Date"
    return df


def choose_input_dir(candidates, required_files):
    best_dir = None
    best_count = -1
    best_missing = list(required_files)
    for directory in candidates:
        existing = [f for f in required_files if (directory / f).exists()]
        if len(existing) > best_count:
            best_dir = directory
            best_count = len(existing)
            best_missing = [f for f in required_files if f not in existing]
    return best_dir, best_missing


def read_panel_csv(path):
    path = Path(path)
    if not path.exists():
        return pd.DataFrame()
    frame = pd.read_csv(path, index_col=0, parse_dates=True)
    frame.index = pd.to_datetime(frame.index).tz_localize(None)
    frame.index.name = "Date"
    return flatten_columns(frame)


def read_table_csv(path):
    path = Path(path)
    if not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path)


def read_json_file(path):
    path = Path(path)
    if not path.exists():
        return {}
    return json.loads(path.read_text())


def read_signal_long(path):
    path = Path(path)
    if not path.exists():
        return pd.DataFrame()
    df = pd.read_csv(path, parse_dates=["Date"])
    df["Date"] = pd.to_datetime(df["Date"]).dt.tz_localize(None)
    return df


def long_signal_to_panel(df, value_col, index=None, columns=None):
    if df.empty or value_col not in df.columns:
        if index is None or columns is None:
            return pd.DataFrame()
        return pd.DataFrame(np.nan, index=index, columns=columns)
    panel = df.pivot(index="Date", columns="Ticker", values=value_col).sort_index()
    panel.index = pd.to_datetime(panel.index).tz_localize(None)
    if index is not None:
        panel = panel.reindex(index)
    if columns is not None:
        panel = panel.reindex(columns=columns)
    panel.index.name = "Date"
    panel.columns.name = None
    return panel


def infer_asset_class_from_description(description):
    description = str(description or "").lower()
    if "reit" in description or "real estate" in description:
        return "REITs"
    if "treasur" in description or "bond" in description or "t-bill" in description or "tips" in description:
        return "Bonds"
    if "gold" in description or "silver" in description or "oil" in description or "commodit" in description or "agriculture" in description:
        return "Commodities"
    if "dollar" in description or "currency" in description or "fx" in description:
        return "FX"
    return "Equities"


def build_asset_class_map(universe_def, universe_metadata, columns):
    asset_class_map = {}
    if isinstance(universe_def.get("asset_class_map"), dict):
        asset_class_map.update(universe_def["asset_class_map"])
    if not universe_metadata.empty:
        meta = universe_metadata.copy()
        meta.columns = [str(c).strip().lower() for c in meta.columns]
        if {"ticker", "asset_class"}.issubset(meta.columns):
            asset_class_map.update(meta.set_index("ticker")["asset_class"].to_dict())
    descriptions = universe_def.get("descriptions", {})
    return {
        ticker: asset_class_map.get(ticker, infer_asset_class_from_description(descriptions.get(ticker)))
        for ticker in columns
    }


def get_proxy_ticker(proxy_mapping, role, fallback=None):
    if not isinstance(proxy_mapping, dict):
        return fallback
    entry = proxy_mapping.get(role, fallback)
    if isinstance(entry, dict):
        return entry.get("ticker", fallback)
    if isinstance(entry, str):
        return entry
    return fallback


def available_tickers(candidates, universe):
    universe = set(universe)
    return [ticker for ticker in candidates if ticker in universe]


def apply_price_lag(panel, extra_lag=0):
    return panel.shift(SIGNAL_LAG_WEEKS + extra_lag)


def trailing_simple_return(prices, lookback, skip=0):
    return prices.shift(skip).div(prices.shift(lookback)) - 1.0


def rolling_zscore(series, window=REGIME_Z_WINDOW, min_periods=None):
    min_periods = min_periods or max(10, window // 2)
    mean = series.rolling(window, min_periods=min_periods).mean()
    std = series.rolling(window, min_periods=min_periods).std()
    return (series - mean) / (std + 1e-12)


def annualized_vol(returns, window, min_periods=None, periods_per_year=WEEKS_PER_YEAR):
    min_periods = min_periods or max(4, window // 2)
    return returns.rolling(window, min_periods=min_periods).std() * np.sqrt(periods_per_year)


def rolling_downside_semivol(simple_returns, window, min_periods=None):
    min_periods = min_periods or max(4, window // 2)
    downside = simple_returns.clip(upper=0.0)
    return downside.rolling(window, min_periods=min_periods).std() * np.sqrt(WEEKS_PER_YEAR)


def rolling_drawdown_frequency(prices, window, threshold=-0.05, min_periods=None):
    min_periods = min_periods or max(4, window // 2)
    rolling_high = prices.rolling(window, min_periods=min_periods).max()
    drawdown = prices.div(rolling_high) - 1.0
    return drawdown.lt(threshold).rolling(window, min_periods=min_periods).mean()


def rolling_drawdown_severity(prices, window, min_periods=None):
    min_periods = min_periods or max(4, window // 2)
    rolling_high = prices.rolling(window, min_periods=min_periods).max()
    drawdown = prices.div(rolling_high) - 1.0
    return drawdown.clip(upper=0.0).abs().rolling(window, min_periods=min_periods).mean()


def rolling_max_drawdown(prices, window, min_periods=None):
    min_periods = min_periods or max(4, window // 2)
    rolling_high = prices.rolling(window, min_periods=min_periods).max()
    drawdown = prices.div(rolling_high) - 1.0
    return drawdown.rolling(window, min_periods=min_periods).min()


def rolling_beta_panel(asset_returns, market_returns, window=BETA_WINDOW, min_periods=BETA_MIN_PERIODS):
    market_var = market_returns.rolling(window, min_periods=min_periods).var()
    beta_parts = {}
    for ticker in asset_returns.columns:
        cov = asset_returns[ticker].rolling(window, min_periods=min_periods).cov(market_returns)
        beta_parts[ticker] = cov.div(market_var)
    beta = pd.DataFrame(beta_parts)
    beta.index = asset_returns.index
    return beta.reindex(columns=asset_returns.columns)


def panel_dict_to_long(panel_map):
    pieces = []
    for name, panel in panel_map.items():
        stacked = panel.stack(dropna=False).rename(name)
        pieces.append(stacked)
    out = pd.concat(pieces, axis=1).reset_index()
    out = out.rename(columns={"level_0": "Date", "level_1": "Ticker"})
    return out


def monthly_snapshot_mask(index):
    month_key = pd.Series(pd.Index(index).to_period("M").astype(str), index=index)
    mask = month_key.ne(month_key.shift(-1))
    if len(mask) > 0:
        mask.iloc[0] = True
    return mask


def average_pairwise_correlation(returns, window=COVARIANCE_WINDOW):
    # Use rolling pairwise-correlation matrices instead of an explicit O(N^2) loop so the diversification-stress feature stays fast enough for repeated research runs.
    def _mean_off_diag(block):
        arr = block.to_numpy(dtype=float, copy=False)
        if arr.ndim != 2 or arr.shape[0] < 2:
            return np.nan
        mask = ~np.eye(arr.shape[0], dtype=bool)
        vals = arr[mask]
        vals = vals[np.isfinite(vals)]
        return float(vals.mean()) if vals.size else np.nan

    frame = pd.DataFrame(returns).replace([np.inf, -np.inf], np.nan)
    out = pd.Series(np.nan, index=frame.index, name="average_pairwise_corr", dtype=float)
    if frame.empty or frame.shape[1] < 2:
        return out
    min_periods = min(window, max(8, window // 2))
    frame = frame.dropna(axis=1, how="all")
    if frame.shape[1] < 2:
        return out
    rolling_corr = frame.rolling(window, min_periods=min_periods).corr()
    if rolling_corr.empty:
        return out
    avg = rolling_corr.groupby(level=0, sort=False).apply(_mean_off_diag)
    avg = avg.reindex(out.index)
    avg.name = "average_pairwise_corr"
    return avg


def compute_strategy_path(weights, next_week_returns, transaction_cost_bps=DEFAULT_COST_BPS):
    weights = weights.reindex(index=next_week_returns.index, columns=next_week_returns.columns).fillna(0.0)
    gross_return = (weights * next_week_returns).sum(axis=1)
    turnover = 0.5 * weights.diff().abs().sum(axis=1)
    if len(turnover) > 0:
        turnover.iloc[0] = np.nan
    cost = turnover.fillna(0.0) * (transaction_cost_bps / 10000.0)
    net_return = gross_return - cost
    wealth = (1.0 + net_return.fillna(0.0)).cumprod()
    drawdown = wealth.div(wealth.cummax()) - 1.0
    return pd.DataFrame(
        {
            "gross_return": gross_return,
            "net_return": net_return,
            "turnover": turnover,
            "cost": cost,
            "wealth": wealth,
            "drawdown": drawdown,
        }
    )


def summary_metrics(return_series, turnover_series=None):
    returns = pd.Series(return_series).dropna()
    if returns.empty:
        return {
            "ann_return": np.nan,
            "ann_vol": np.nan,
            "sharpe": np.nan,
            "max_drawdown": np.nan,
            "calmar": np.nan,
            "hit_rate": np.nan,
            "avg_weekly_turnover": np.nan,
            "observations": 0,
        }
    wealth = (1.0 + returns).cumprod()
    drawdown = wealth.div(wealth.cummax()) - 1.0
    ann_return = wealth.iloc[-1] ** (WEEKS_PER_YEAR / len(returns)) - 1.0
    ann_vol = returns.std(ddof=1) * np.sqrt(WEEKS_PER_YEAR)
    sharpe = np.nan if ann_vol == 0 or pd.isna(ann_vol) else ann_return / ann_vol
    max_dd = drawdown.min()
    calmar = np.nan if max_dd == 0 or pd.isna(max_dd) else ann_return / abs(max_dd)
    return {
        "ann_return": ann_return,
        "ann_vol": ann_vol,
        "sharpe": sharpe,
        "max_drawdown": max_dd,
        "calmar": calmar,
        "hit_rate": (returns > 0).mean(),
        "avg_weekly_turnover": np.nan if turnover_series is None else pd.Series(turnover_series).mean(),
        "observations": int(len(returns)),
    }


def compute_datewise_ic(signal_panel, forward_return_panel):
    rows = []
    for date in signal_panel.index.intersection(forward_return_panel.index):
        joint = pd.concat(
            [signal_panel.loc[date].rename("signal"), forward_return_panel.loc[date].rename("fwd")],
            axis=1,
        ).dropna()
        if len(joint) < 5:
            continue
        rows.append({"Date": date, "ic": joint["signal"].corr(joint["fwd"], method="spearman")})
    if not rows:
        return pd.Series(dtype=float)
    out = pd.DataFrame(rows).set_index("Date")["ic"].sort_index()
    out.name = "ic"
    return out


def compounded_forward_return(simple_returns, horizon):
    gross = 1.0 + simple_returns
    out = pd.DataFrame(1.0, index=simple_returns.index, columns=simple_returns.columns)
    for step in range(1, horizon + 1):
        out = out * gross.shift(-step)
    return out - 1.0


## 2. Research Anchors for Layer 2B

This notebook focuses on practical risk and regime ideas that are useful for a weekly ETF allocator.

**Core references used here**

- Moreira and Muir, *Volatility-Managed Portfolios*:
  [NBER working paper](https://www.nber.org/papers/w22208)
- Hurst, Ooi, and Pedersen, *A Century of Evidence on Trend-Following Investing*:
  [AQR article](https://www.aqr.com/Insights/Research/Journal-Article/A-Century-of-Evidence-on-Trend-Following-Investing)
- Keller and Keuning, *Defensive Asset Allocation*:
  [Accessible PDF mirror](https://assets.super.so/e46b77e7-ee08-445e-b43f-4ffd88ae0a0e/files/ff84541a-07f5-4a71-8476-b7e788211241.pdf)
- Ledoit and Wolf, *Honey, I Shrunk the Sample Covariance Matrix*:
  [Paper page](https://econ-papers.upf.edu/papers/691.pdf)

**How the notebook interprets this literature**

- volatility and covariance estimates are used as **inputs**, not as a claim that risk can be forecast perfectly
- breadth, volatility, and correlation stress are used as **interpretable diagnostics**
- regime labels are kept deliberately transparent so they can later be audited or overridden in Layer 3


In [ ]:
data_hub_dir, missing_hub = choose_input_dir(DATA_HUB_CANDIDATES, REQUIRED_DATA_HUB)
if missing_hub:
    raise FileNotFoundError(
        f"Missing required data-hub inputs: {missing_hub}. Run 01_data_hub.ipynb first."
    )

layer1_dir, missing_layer1 = choose_input_dir(LAYER1_CANDIDATES, OPTIONAL_LAYER1)

weekly_prices = read_panel_csv(data_hub_dir / "weekly_prices.csv")
weekly_log_returns = read_panel_csv(data_hub_dir / "weekly_returns.csv")
weekly_simple_returns = np.expm1(weekly_log_returns).reindex_like(weekly_prices)
next_week_returns = weekly_simple_returns.shift(-1)
daily_log_returns = read_panel_csv(data_hub_dir / "daily_returns.csv")
macro_weekly = read_panel_csv(data_hub_dir / "macro_weekly.csv")
vix_term = read_panel_csv(data_hub_dir / "vix_term_structure.csv")
universe_def = read_json_file(data_hub_dir / "universe.json")
universe_metadata = read_table_csv(data_hub_dir / "universe_metadata.csv")
proxy_mapping = read_json_file(data_hub_dir / "proxy_mapping.json")
market_proxy_weekly = read_panel_csv(data_hub_dir / "market_proxy_weekly.csv")
benchmark_returns = read_panel_csv(data_hub_dir / "benchmark_returns_weekly.csv")

research_universe = universe_def.get("full", universe_def.get("all", list(weekly_prices.columns)))
research_universe = [ticker for ticker in research_universe if ticker in weekly_prices.columns]
weekly_prices = weekly_prices.reindex(columns=research_universe)
weekly_simple_returns = weekly_simple_returns.reindex(index=weekly_prices.index, columns=research_universe)
next_week_returns = next_week_returns.reindex(index=weekly_prices.index, columns=research_universe)
if not daily_log_returns.empty:
    daily_log_returns = daily_log_returns.reindex(columns=research_universe)

asset_class_map = build_asset_class_map(universe_def, universe_metadata, research_universe)
asset_class_series = pd.Series(asset_class_map).reindex(research_universe)

market_proxy = get_proxy_ticker(proxy_mapping, "market_proxy", fallback="SPY")
cash_proxy = get_proxy_ticker(proxy_mapping, "cash_proxy", fallback="SHY" if "SHY" in research_universe else "BIL")
duration_proxy = get_proxy_ticker(
    proxy_mapping,
    "intermediate_duration_proxy",
    fallback=get_proxy_ticker(proxy_mapping, "duration_proxy", fallback="IEF"),
)

if not market_proxy_weekly.empty and "market_proxy_log_return" in market_proxy_weekly.columns:
    market_log_return = market_proxy_weekly["market_proxy_log_return"].reindex(weekly_prices.index)
elif market_proxy in weekly_log_returns.columns:
    market_log_return = weekly_log_returns[market_proxy].reindex(weekly_prices.index)
elif not benchmark_returns.empty and market_proxy in benchmark_returns.columns:
    market_log_return = benchmark_returns[market_proxy].reindex(weekly_prices.index)
else:
    raise ValueError("No explicit market proxy return series is available.")

market_simple_return = np.expm1(market_log_return)

broad_risk_assets = available_tickers(
    ["SPY", "QQQ", "IWM", "EFA", "VEA", "VWO", "EWJ", "VNQ", "HYG", "LQD", "GLD", "PDBC", "DBA", "TLT"],
    research_universe,
)
canary_assets = available_tickers(["VWO", "HYG", "VNQ", "EFA", "PDBC"], research_universe)

regime_features = read_panel_csv(layer1_dir / "regime_features.csv") if layer1_dir else pd.DataFrame()
signal_xsmom = read_signal_long(layer1_dir / "signal_xsmom.csv") if layer1_dir else pd.DataFrame()
signal_reversal = read_signal_long(layer1_dir / "signal_reversal.csv") if layer1_dir else pd.DataFrame()
signal_multi_mom = read_signal_long(layer1_dir / "signal_multi_horizon_mom.csv") if layer1_dir else pd.DataFrame()
signal_quality = read_signal_long(layer1_dir / "signal_quality.csv") if layer1_dir else pd.DataFrame()

xsmom_panel = long_signal_to_panel(signal_xsmom, "xsmom_score_tradable", index=weekly_prices.index, columns=research_universe)
reversal_panel = long_signal_to_panel(signal_reversal, "reversal_1w_score_tradable", index=weekly_prices.index, columns=research_universe)
multi_mom_panel = long_signal_to_panel(signal_multi_mom, "multi_mom_invvol_score_tradable", index=weekly_prices.index, columns=research_universe)
quality_panel = long_signal_to_panel(signal_quality, "quality_score_tradable", index=weekly_prices.index, columns=research_universe)

audit_table = pd.DataFrame(
    [
        {"item": "Data hub dir", "value": str(data_hub_dir.resolve())},
        {"item": "Layer 1 dir", "value": None if layer1_dir is None else str(layer1_dir.resolve())},
        {"item": "Research universe size", "value": len(research_universe)},
        {"item": "Market proxy", "value": market_proxy},
        {"item": "Cash proxy", "value": cash_proxy},
        {"item": "Duration proxy", "value": duration_proxy},
        {"item": "Broad risk assets", "value": ", ".join(broad_risk_assets)},
    ]
)

print("Layer 2B input audit")
print("=" * 90)
print(audit_table.to_string(index=False))
print()
print("Asset-class counts")
print(asset_class_series.value_counts().to_string())


## 3. Rolling Risk Estimates

Risk estimates are not forecasts of the future. They are state variables that help us avoid pretending every environment deserves the same gross exposure.

The notebook builds:

- weekly realized volatility
- daily-based realized volatility when daily returns are available
- downside semivolatility
- drawdown frequency, drawdown severity, and a rolling max-drawdown proxy
- rolling beta to the explicit market benchmark


In [ ]:
risk_panel_map = {}

for window in VOL_WINDOWS:
    risk_panel_map[f"weekly_vol_{window}w_observed"] = annualized_vol(weekly_simple_returns, window=window)
    risk_panel_map[f"weekly_vol_{window}w_tradable"] = apply_price_lag(risk_panel_map[f"weekly_vol_{window}w_observed"])

if not daily_log_returns.empty:
    daily_simple_returns = np.expm1(daily_log_returns)
    for window in DAILY_VOL_WINDOWS:
        daily_vol = annualized_vol(daily_simple_returns, window=window, periods_per_year=TRADING_DAYS_PER_YEAR)
        risk_panel_map[f"daily_vol_{window}d_observed"] = daily_vol.resample("W-FRI").last().reindex(weekly_prices.index)
        risk_panel_map[f"daily_vol_{window}d_tradable"] = apply_price_lag(risk_panel_map[f"daily_vol_{window}d_observed"])

risk_panel_map["downside_semivol_26w_observed"] = rolling_downside_semivol(weekly_simple_returns, window=26)
risk_panel_map["downside_semivol_26w_tradable"] = apply_price_lag(risk_panel_map["downside_semivol_26w_observed"])
risk_panel_map["drawdown_frequency_52w_observed"] = rolling_drawdown_frequency(weekly_prices, window=DRAWDOWN_WINDOW)
risk_panel_map["drawdown_frequency_52w_tradable"] = apply_price_lag(risk_panel_map["drawdown_frequency_52w_observed"])
risk_panel_map["drawdown_severity_52w_observed"] = rolling_drawdown_severity(weekly_prices, window=DRAWDOWN_WINDOW)
risk_panel_map["drawdown_severity_52w_tradable"] = apply_price_lag(risk_panel_map["drawdown_severity_52w_observed"])
risk_panel_map["max_drawdown_52w_observed"] = rolling_max_drawdown(weekly_prices, window=DRAWDOWN_WINDOW)
risk_panel_map["max_drawdown_52w_tradable"] = apply_price_lag(risk_panel_map["max_drawdown_52w_observed"])

beta_panel = rolling_beta_panel(weekly_simple_returns, market_simple_return, window=BETA_WINDOW, min_periods=BETA_MIN_PERIODS)
risk_panel_map["beta_to_market_observed"] = beta_panel
risk_panel_map["beta_to_market_tradable"] = apply_price_lag(beta_panel)

risk_metrics_long = panel_dict_to_long(risk_panel_map)
risk_metrics_long["asset_class"] = risk_metrics_long["Ticker"].map(asset_class_map)

fig, axes = plt.subplots(2, 2, figsize=(15, 9))
example_assets = [ticker for ticker in [market_proxy, "TLT", "GLD", "HYG"] if ticker in research_universe][:4]
if example_assets:
    risk_panel_map["weekly_vol_26w_tradable"][example_assets].plot(ax=axes[0, 0], lw=1.5)
    axes[0, 0].set_title("Weekly realized vol (26w, tradable)")
    risk_panel_map["downside_semivol_26w_tradable"][example_assets].plot(ax=axes[0, 1], lw=1.5)
    axes[0, 1].set_title("Downside semivol (26w, tradable)")
    risk_panel_map["drawdown_severity_52w_tradable"][example_assets].plot(ax=axes[1, 0], lw=1.5)
    axes[1, 0].set_title("Drawdown severity (52w, tradable)")
    risk_panel_map["beta_to_market_tradable"][example_assets].plot(ax=axes[1, 1], lw=1.5)
    axes[1, 1].set_title(f"Rolling beta to {market_proxy} (tradable)")
plt.tight_layout()
plt.show()


## 4. Covariance Inputs for Later Layer 3 Work

Full portfolio optimization is not the goal here. But later portfolio construction will still need covariance inputs that are:

- clean
- reproducible
- consistent with the weekly ETF research frame

This notebook therefore saves:

- sample rolling covariance snapshots
- Ledoit-Wolf shrinkage covariance snapshots when the estimator is available
- latest wide matrices for immediate inspection or later use


In [ ]:
snapshot_mask = monthly_snapshot_mask(weekly_prices.index)
snapshot_dates = weekly_prices.index[snapshot_mask]

covariance_rows = []
lw_rows = []
latest_sample_cov = pd.DataFrame()
latest_lw_cov = pd.DataFrame()

for date in snapshot_dates:
    trailing = weekly_simple_returns.loc[:date].tail(COVARIANCE_WINDOW)
    trailing = trailing.dropna(axis=1, thresh=max(8, int(0.7 * COVARIANCE_WINDOW)))
    trailing = trailing.dropna(axis=0, how="all")
    if trailing.shape[1] < 2 or trailing.shape[0] < max(8, COVARIANCE_WINDOW // 2):
        continue

    sample_cov = trailing.cov()
    latest_sample_cov = sample_cov.copy()
    for left in sample_cov.index:
        for right in sample_cov.columns:
            covariance_rows.append(
                {
                    "Date": date,
                    "asset_i": left,
                    "asset_j": right,
                    "covariance": sample_cov.loc[left, right],
                    "estimator": "sample",
                }
            )

    complete = trailing.dropna(axis=1, thresh=trailing.shape[0]).dropna(axis=0, how="any")
    if LedoitWolf is not None and complete.shape[1] >= 2 and complete.shape[0] >= max(8, COVARIANCE_WINDOW // 2):
        lw = LedoitWolf().fit(complete.values)
        lw_cov = pd.DataFrame(lw.covariance_, index=complete.columns, columns=complete.columns)
        latest_lw_cov = lw_cov.copy()
        for left in lw_cov.index:
            for right in lw_cov.columns:
                lw_rows.append(
                    {
                        "Date": date,
                        "asset_i": left,
                        "asset_j": right,
                        "covariance": lw_cov.loc[left, right],
                        "estimator": "ledoit_wolf",
                    }
                )

covariance_long = pd.DataFrame(covariance_rows)
covariance_lw_long = pd.DataFrame(lw_rows)


## 5. Regime Features, Classification, and Defensive Overlay Logic

A useful regime engine should be:

- interpretable
- based on features that are actually available
- conservative about timing
- simple enough that later notebooks can trust and reuse it

The regime engine below blends:

- Layer 1 macro / VIX conditioning features
- market realized volatility
- drawdown severity
- breadth of positive momentum across risky ETFs
- average pairwise correlation as a diversification-stress proxy

It then produces:

- a continuous `risk_regime_score`
- discrete `risk_state` labels
- a `signal_environment` label such as `trend_friendly` or `reversal_friendly`
- a reusable defensive overlay multiplier

A small implementation note: this notebook's `target_vol_multiplier` uses market-proxy volatility as a regime scalar, so its ceiling is intentionally a little looser than Layer 3's final allocator cap. Layer 3 re-estimates volatility at the portfolio level before deciding final exposure.


In [ ]:
regime_score = pd.DataFrame(index=weekly_prices.index)
regime_score.index.name = "Date"

market_vol_13w = annualized_vol(market_simple_return.to_frame(market_proxy), window=13)[market_proxy]
market_vol_13w_tradable = apply_price_lag(market_vol_13w.to_frame("market_vol_13w"))["market_vol_13w"]
market_drawdown = rolling_drawdown_severity(weekly_prices[[market_proxy]], window=DRAWDOWN_WINDOW)[market_proxy]
market_drawdown_tradable = apply_price_lag(market_drawdown.to_frame("market_drawdown"))["market_drawdown"]

raw_breadth = trailing_simple_return(weekly_prices[canary_assets], lookback=52, skip=4).gt(0).mean(axis=1) if canary_assets else pd.Series(np.nan, index=weekly_prices.index)
breadth_tradable = apply_price_lag(raw_breadth.to_frame("breadth"))["breadth"]

avg_corr_observed = average_pairwise_correlation(weekly_simple_returns[broad_risk_assets], window=COVARIANCE_WINDOW) if broad_risk_assets else pd.Series(np.nan, index=weekly_prices.index)
avg_corr_tradable = apply_price_lag(avg_corr_observed.to_frame("avg_corr"))["avg_corr"]

regime_score["market_vol_risk_off_z"] = rolling_zscore(market_vol_13w_tradable)
regime_score["market_drawdown_risk_off_z"] = rolling_zscore(market_drawdown_tradable)
regime_score["breadth_risk_off_z"] = -rolling_zscore(breadth_tradable)
regime_score["avg_corr_risk_off_z"] = rolling_zscore(avg_corr_tradable)

if not regime_features.empty:
    for col in ["macro_risk_score_tradable", "vix_level_z_tradable", "vix_slope_risk_off_z_tradable", "google_fear_z_tradable"]:
        if col in regime_features.columns:
            regime_score[col] = regime_features[col].reindex(weekly_prices.index)

component_cols = [col for col in regime_score.columns if col != "risk_regime_score"]
regime_score["risk_regime_score"] = regime_score[component_cols].mean(axis=1)
regime_score["risk_state"] = pd.Series(
    np.where(
        regime_score["risk_regime_score"] >= 0.75,
        "stressed",
        np.where(regime_score["risk_regime_score"] <= -0.50, "calm", "neutral"),
    ),
    index=regime_score.index,
    dtype="object",
)
regime_score["signal_environment"] = pd.Series(
    np.where(
        (breadth_tradable >= 0.60) & regime_score["risk_state"].eq("calm"),
        "trend_friendly",
        np.where(
            (breadth_tradable <= 0.40) | regime_score["risk_state"].eq("stressed"),
            "reversal_friendly",
            "mixed",
        ),
    ),
    index=regime_score.index,
    dtype="object",
)

overlay_multiplier = pd.Series(
    np.where(
        regime_score["risk_state"].eq("stressed"),
        0.30,
        np.where(regime_score["risk_state"].eq("calm"), 1.00, 0.65),
    ),
    index=regime_score.index,
    dtype=float,
    name="overlay_multiplier",
)
if market_vol_13w_tradable.notna().sum() > 0:
    # This is a market-vol regime proxy rather than a final portfolio-vol estimate, so the ceiling is intentionally looser than Layer 3's allocator cap.
    target_vol_multiplier = (REGIME_TARGET_VOL_ANN / market_vol_13w_tradable.replace(0, np.nan)).clip(lower=REGIME_TARGET_VOL_FLOOR, upper=REGIME_TARGET_VOL_CEIL).fillna(0.75)
else:
    target_vol_multiplier = pd.Series(1.0, index=regime_score.index, name="target_vol_multiplier")

overlay_states = pd.DataFrame(index=weekly_prices.index)
overlay_states.index.name = "Date"
overlay_states["risk_state"] = regime_score["risk_state"]
overlay_states["signal_environment"] = regime_score["signal_environment"]
overlay_states["overlay_multiplier"] = np.minimum(overlay_multiplier, target_vol_multiplier)
overlay_states["target_vol_multiplier"] = target_vol_multiplier
overlay_states["defensive_weight"] = 1.0 - overlay_states["overlay_multiplier"]
overlay_states["defensive_asset"] = cash_proxy
overlay_states["secondary_defensive_asset"] = duration_proxy

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
regime_score["risk_regime_score"].plot(ax=axes[0], color="steelblue", lw=1.6)
axes[0].axhline(0.75, color="darkred", lw=1, ls="--")
axes[0].axhline(-0.50, color="darkgreen", lw=1, ls="--")
axes[0].set_title("Composite risk regime score")
breadth_tradable.plot(ax=axes[1], color="darkorange", lw=1.6)
axes[1].axhline(0.60, color="darkgreen", lw=1, ls="--")
axes[1].axhline(0.40, color="darkred", lw=1, ls="--")
axes[1].set_title("Canary / risky breadth (share of positive momentum)")
overlay_states["overlay_multiplier"].plot(ax=axes[2], color="purple", lw=1.6)
axes[2].set_title("Defensive overlay multiplier")
plt.tight_layout()
plt.show()


## 6. Regime Validation

A regime engine is only useful if it changes something real:

- do momentum-like signals behave better in calm or trend-friendly states?
- does reversal look better when the environment is stressed or whipsaw-prone?
- does a simple defensive overlay actually reduce drawdown instead of just hiding in cash forever?


In [ ]:
validation_rows = []

four_week_forward = compounded_forward_return(weekly_simple_returns, horizon=4)
signal_validation_map = {
    "xsmom_global": xsmom_panel,
    "reversal_1w_global": reversal_panel,
    "multi_mom_invvol": multi_mom_panel,
    "quality_proxy": quality_panel,
}

for signal_name, panel in signal_validation_map.items():
    if panel.empty or panel.notna().sum().sum() == 0:
        continue
    ic_series = compute_datewise_ic(panel, four_week_forward)
    if ic_series.empty:
        continue
    joined = pd.DataFrame({"ic": ic_series}).join(overlay_states[["risk_state", "signal_environment"]], how="left")
    grouped = joined.groupby("risk_state")["ic"].agg(["mean", "median", "count"]).reset_index()
    grouped["signal_name"] = signal_name
    grouped["validation_type"] = "ic_by_risk_state"
    validation_rows.append(grouped)

risky_sleeve_weights = pd.DataFrame(0.0, index=weekly_prices.index, columns=weekly_prices.columns)
if broad_risk_assets:
    risky_sleeve_weights.loc[:, broad_risk_assets] = 1.0 / len(broad_risk_assets)
risky_sleeve_path = compute_strategy_path(risky_sleeve_weights, next_week_returns, transaction_cost_bps=DEFAULT_COST_BPS)

overlay_weights = risky_sleeve_weights.copy()
if cash_proxy in overlay_weights.columns:
    overlay_weights[broad_risk_assets] = overlay_weights[broad_risk_assets].mul(overlay_states["overlay_multiplier"], axis=0)
    overlay_weights[cash_proxy] = 1.0 - overlay_weights[broad_risk_assets].sum(axis=1)
overlay_path = compute_strategy_path(overlay_weights.fillna(0.0), next_week_returns, transaction_cost_bps=DEFAULT_COST_BPS)

overlay_summary = pd.DataFrame(
    [
        {"strategy_name": "risky_sleeve_unfiltered", **summary_metrics(risky_sleeve_path["net_return"], risky_sleeve_path["turnover"])},
        {"strategy_name": "risky_sleeve_with_overlay", **summary_metrics(overlay_path["net_return"], overlay_path["turnover"])},
    ]
)

trend_strategy_weights = pd.DataFrame(0.0, index=weekly_prices.index, columns=weekly_prices.columns)
if not multi_mom_panel.empty and broad_risk_assets:
    positive_trend = multi_mom_panel[broad_risk_assets].gt(0).astype(float)
    denom = positive_trend.sum(axis=1).replace(0, np.nan)
    trend_strategy_weights[broad_risk_assets] = positive_trend.div(denom, axis=0).fillna(0.0)
    if cash_proxy in trend_strategy_weights.columns:
        trend_strategy_weights[cash_proxy] = 1.0 - trend_strategy_weights[broad_risk_assets].sum(axis=1)
trend_path = compute_strategy_path(trend_strategy_weights.fillna(0.0), next_week_returns, transaction_cost_bps=DEFAULT_COST_BPS)
trend_by_regime = pd.DataFrame({"return": trend_path["net_return"]}).join(overlay_states[["risk_state", "signal_environment"]], how="left")
trend_by_regime_summary = trend_by_regime.groupby("risk_state")["return"].agg(["mean", "std", "count"]).reset_index()
trend_by_regime_summary["validation_type"] = "trend_return_by_risk_state"

validation_summary = pd.concat(validation_rows + [trend_by_regime_summary], ignore_index=True) if validation_rows else trend_by_regime_summary.copy()

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
risky_sleeve_path["wealth"].plot(ax=axes[0], lw=1.6, label="risky_sleeve_unfiltered")
overlay_path["wealth"].plot(ax=axes[0], lw=1.6, label="risky_sleeve_with_overlay")
axes[0].set_title("Overlay validation: wealth curves")
axes[0].legend(loc="upper left")
risky_sleeve_path["drawdown"].plot(ax=axes[1], lw=1.4, label="risky_sleeve_unfiltered")
overlay_path["drawdown"].plot(ax=axes[1], lw=1.4, label="risky_sleeve_with_overlay")
axes[1].set_title("Overlay validation: drawdowns")
axes[1].legend(loc="lower left")
plt.tight_layout()
plt.show()

print("Overlay comparison")
print(overlay_summary.round(4).to_string(index=False))
print()
print("Signal / regime interaction summary")
print(validation_summary.round(4).to_string(index=False))


## 7. Save Outputs and Layer 3 Hand-Off

The outputs saved here are meant to be reused by both:

- Layer 2A strategy logic on reruns, when you want a richer regime source
- later Layer 3 sizing / optimization work, where covariance and risk-budget inputs become unavoidable


In [ ]:
risk_metrics_path = OUTPUT_DIR / "risk_metrics_panel.csv"
regime_score_path = OUTPUT_DIR / "regime_score.csv"
regime_states_path = OUTPUT_DIR / "regime_states.csv"
overlay_path_csv = OUTPUT_DIR / "defensive_overlay_states.csv"
covariance_path = OUTPUT_DIR / "covariance_snapshots_long.csv"
covariance_lw_path = OUTPUT_DIR / "covariance_ledoit_wolf_snapshots_long.csv"
latest_cov_path = OUTPUT_DIR / "latest_covariance_matrix.csv"
latest_cov_lw_path = OUTPUT_DIR / "latest_ledoit_wolf_covariance_matrix.csv"
overlay_validation_path = OUTPUT_DIR / "overlay_validation_table.csv"
validation_summary_path = OUTPUT_DIR / "signal_regime_interaction_summary.csv"
manifest_path = OUTPUT_DIR / "layer2_manifest.json"

risk_metrics_long.to_csv(risk_metrics_path, index=False)
regime_score.to_csv(regime_score_path)
overlay_states.to_csv(regime_states_path)
overlay_states.to_csv(overlay_path_csv)
covariance_long.to_csv(covariance_path, index=False)
covariance_lw_long.to_csv(covariance_lw_path, index=False)
latest_sample_cov.to_csv(latest_cov_path)
latest_lw_cov.to_csv(latest_cov_lw_path)
overlay_summary.to_csv(overlay_validation_path, index=False)
validation_summary.to_csv(validation_summary_path, index=False)

manifest_records = [
    {
        "strategy_name": "risk_metrics_panel",
        "notebook_origin": "04_layer2b_risk_regime_engine.ipynb",
        "type": "risk_estimate",
        "required_inputs": ["weekly_prices.csv", "weekly_returns.csv", "daily_returns.csv"],
        "rebalance_frequency": "weekly",
        "lag_convention": "Price-based risk metrics are lagged one week for decision use.",
        "output_files": ["risk_metrics_panel.csv"],
        "caveats": "These are state variables, not perfect forecasts of future risk.",
    },
    {
        "strategy_name": "regime_engine",
        "notebook_origin": "04_layer2b_risk_regime_engine.ipynb",
        "type": "regime",
        "required_inputs": ["regime_features.csv", "weekly_returns.csv", "vix_term_structure.csv", "macro_weekly.csv"],
        "rebalance_frequency": "weekly",
        "lag_convention": "Uses tradable Layer 1 regime features plus lagged price-based breadth / vol diagnostics.",
        "output_files": ["regime_score.csv", "regime_states.csv", "defensive_overlay_states.csv"],
        "caveats": "Transparent equal-weight style composite rather than an optimized hidden-state model.",
    },
    {
        "strategy_name": "covariance_inputs",
        "notebook_origin": "04_layer2b_risk_regime_engine.ipynb",
        "type": "risk_estimate",
        "required_inputs": ["weekly_returns.csv"],
        "rebalance_frequency": "monthly snapshots",
        "lag_convention": "Built from trailing weekly returns only.",
        "output_files": [
            "covariance_snapshots_long.csv",
            "covariance_ledoit_wolf_snapshots_long.csv",
            "latest_covariance_matrix.csv",
            "latest_ledoit_wolf_covariance_matrix.csv",
        ],
        "caveats": "Shrinkage estimator uses only the subset of assets with enough complete trailing data.",
    },
    {
        "strategy_name": "defensive_overlay_validation",
        "notebook_origin": "04_layer2b_risk_regime_engine.ipynb",
        "type": "overlay",
        "required_inputs": ["weekly_returns.csv", "regime_score.csv"],
        "rebalance_frequency": "weekly",
        "lag_convention": "Overlay multiplier is saved in tradable form.",
        "output_files": ["overlay_validation_table.csv", "signal_regime_interaction_summary.csv"],
        "caveats": "Validation is intentionally simple and should be treated as directional evidence, not final proof.",
    },
]
manifest_path.write_text(json.dumps(manifest_records, indent=2))

print(f"Saved: {risk_metrics_path}")
print(f"Saved: {regime_score_path}")
print(f"Saved: {regime_states_path}")
print(f"Saved: {covariance_path}")
print(f"Saved: {covariance_lw_path}")
print(f"Saved: {manifest_path}")


## 8. Final Summary

1. **What was implemented**
   Rolling weekly and daily-based volatility estimates, downside risk measures, drawdown diagnostics, rolling beta, covariance snapshots, a transparent composite regime score, discrete regime states, and a reusable defensive overlay.

2. **Why those Layer 2B components matter**
   They give the strategy layer a disciplined way to change exposure when the environment is calm versus stressed, and they prepare the risk inputs that later Layer 3 portfolio construction will need.

3. **Which strategies look most robust so far**
   The notebook is deliberately not ranking full strategies, but the validation is most supportive of slower, breadth-aware trend sleeves rather than fast mean-reversion logic.

4. **Which risk / regime features appear genuinely useful**
   The most useful features so far are the composite macro / VIX risk score, realized market volatility, drawdown severity, breadth, and average pairwise correlation as a diversification-stress proxy.

5. **Which ideas are still too data-limited or experimental**
   Hidden-state models, richer crash-prediction logic, and deeper cross-asset liquidity proxies are still more experimental here than the simpler baseline regime labels.

6. **Which Layer 2 outputs are best candidates to feed into Layer 3 later**
   The strongest Layer 3 hand-off artifacts are the risk metrics panel, regime states, overlay multipliers, and the covariance snapshots, especially the shrinkage covariance when enough trailing data are available.
